<a href="https://colab.research.google.com/github/NITHIN-aiml69/python-project-sem-3/blob/main/catboost_linear_0.4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import pandas as pd

# --- 1. Load data ---
df1 = pd.read_csv('Tuesday-WorkingHours.pcap_ISCX.csv', low_memory=True)
df2 = pd.read_csv('Wednesday-workingHours.pcap_ISCX.csv', low_memory=True)
df3 = pd.read_csv('Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', low_memory=True)

dataset = pd.concat([df1, df2, df3], ignore_index=True)
dataset.columns = dataset.columns.str.strip()

X = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1]

X = X.apply(pd.to_numeric, errors='coerce')
X.replace([np.inf, -np.inf], np.nan, inplace=True)

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X = imputer.fit_transform(X)

from sklearn.preprocessing import LabelEncoder
labelencoder_y = LabelEncoder()
y = labelencoder_y.fit_transform(y)

# --- 2. SUBSAMPLE (this was missing -> caused the RAM crash) ---
# Kernel PCA needs an n x n matrix in memory, so the full ~1.3M rows
# is impossible. This keeps 15,000 rows with the same class balance.
from sklearn.model_selection import train_test_split
X, _, y, _ = train_test_split(X, y, train_size=15000, random_state=0, stratify=y)

# --- 3. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,        # change to 0.2 or 0.6 for other runs
    random_state=0,
    stratify=y
)

# --- 4. Kernel PCA ---
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(n_components=2, kernel='linear', gamma=15)   # swap kernel here for other runs
X_train = kpca.fit_transform(X_train)
X_test = kpca.transform(X_test)

# --- 5. Classifier (swap this block for each algorithm) ---
from catboost import CatBoostClassifier
classifier = CatBoostClassifier(verbose=0, random_state=0)

classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# --- 6. Metrics ---
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n")
print(cm)

print("\nAccuracy : {:.4f}".format(accuracy_score(y_test, y_pred)))
print("\nPrecision : {:.4f}".format(precision_score(y_test, y_pred, average='weighted')))
print("\nRecall : {:.4f}".format(recall_score(y_test, y_pred, average='weighted')))
print("\nF1 Score : {:.4f}".format(f1_score(y_test, y_pred, average='weighted')))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Confusion Matrix:

[[4667    1   85    1    3   12    0    0    0]
 [  21   22    3    0    0    1    0    0    0]
 [  92    0  966    1    0    0    0    0    0]
 [   4    0    0   21    0    0    0    0    0]
 [   9    0    0    0   17    0    0    0    0]
 [  14    0    0    0    0   23    0    0    0]
 [  13    0    0    0    0    1   13    0    0]
 [   2    0    0    0    0    0    0    2    3]
 [   2    0    0    0    0    0    0    0    1]]

Accuracy : 0.9553

Precision : 0.9553

Recall : 0.9553

F1 Score : 0.9539

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.98      0.97      4769
           1       0.96      0.47      0.63        47
           2       0.92      0.91      0.91      1059
           3       0.91      0.84      0.88        25
           4       0.85      0.65      0.74        26
           5       0.62      0.62      0.62        37
           7       1.00      0.48      0.65        27
           8   

In [8]:
!pip install catboost